# Phase 3: Evaluation extras + demo data

Calibration (3 models, same comments) · routing · error review · HateCheck bias test · speed (one-step vs two-step, GPU vs CPU) · optional Gemini race · all demo data files.

**Settings:** GPU T4 ×2 · Internet On · Inputs: **Toxic Comment Classification Challenge** + **your Phase 2c notebook's output** (Add Input → Your Work → Notebooks) · Secrets: **HF_TOKEN** (+ **GEMINI_API_KEY** for the race).
**Run with Save Version → Save & Run All** (~1 h).

In [1]:
# ==== CONFIG ====
FT_REPO = "Vinay57/laya-jigsaw-moderation"   # main = merged model; fine-tuned-only version is in its history
BASE_REPO = "convaiinnovations/laya"
BATCH = 64
CASCADE_TAU = 0.1        # two-step check: ask the 5 detail questions only if P(toxic) >= this
RUN_RACE = True          # needs Kaggle secret GEMINI_API_KEY; skipped automatically if missing
RACE_N = 100
GEMINI_MODEL = "gemini-2.5-flash"   # use any Flash model listed in your Google AI Studio
GEMINI_RPM = 10          # stay under your free-tier requests-per-minute

In [2]:
!pip install -q -U laya detoxify
import os, glob, json, time, math, numpy as np, pandas as pd, torch, laya
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 170); pd.set_option("display.width", 250)
OUT = "/kaggle/working/demo_data"; PLOTS = "/kaggle/working/plots"
os.makedirs(OUT, exist_ok=True); os.makedirs(PLOTS, exist_ok=True)
print("laya", laya.__version__, "| CUDA", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 6.2 MB/s eta 0:00:00
laya 0.3.20 | CUDA True


## 1. Data (same 10k test sample as before)

In [3]:
LABELS = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
QUESTIONS = {
  'toxic':         {'type':'noul','instructions':'Is this comment rude, disrespectful or unreasonable?'},
  'severe_toxic':  {'type':'noul','instructions':'Is this comment extremely hateful, aggressive or disrespectful?'},
  'obscene':       {'type':'noul','instructions':'Does this comment contain obscene or vulgar language?'},
  'threat':        {'type':'noul','instructions':'Does this comment threaten to harm someone?'},
  'insult':        {'type':'noul','instructions':'Does this comment insult a person?'},
  'identity_hate': {'type':'noul','instructions':'Does this comment attack someone for their identity (race, religion, gender, sexuality, etc.)?'},
}
MAX_CHARS = 1200
BASE = os.path.dirname(glob.glob('/kaggle/input/**/train.csv*', recursive=True)[0])
def load(name):
    for ext in ['.csv', '.csv.zip']:
        p = f'{BASE}/{name}{ext}'
        if os.path.exists(p): return pd.read_csv(p)
    raise FileNotFoundError(name)
test = load('test').merge(load('test_labels'), on='id')
test = test[test['toxic'] != -1].reset_index(drop=True)
sample = test.sample(10000, random_state=42)
ys = sample[LABELS].values
print(len(test), len(sample))

63978 10000


## 2. Load the three models (zero-shot, fine-tuned, merged)

In [4]:
from huggingface_hub import snapshot_download, HfApi
from kaggle_secrets import UserSecretsClient
from safetensors.torch import load_file
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_model, build_sequence, QTYPES, ece_score
from sklearn.metrics import roc_auc_score

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
NOUL = QTYPES["noul"]
IGN = ["multilingual/*", "typed-decisions/*", "eval/*", "assets/*"]

api = HfApi(token=HF_TOKEN)
ft_commit = next(c.commit_id for c in api.list_repo_commits(FT_REPO) if c.title.startswith("Jigsaw fine-tune"))
DIRS = {"merged": snapshot_download(FT_REPO, token=HF_TOKEN, ignore_patterns=IGN),
        "fine_tuned": snapshot_download(FT_REPO, revision=ft_commit, token=HF_TOKEN, ignore_patterns=IGN),
        "zero_shot": snapshot_download(BASE_REPO, ignore_patterns=IGN)}
for d in DIRS.values(): _fix_tokenizer_config(d)

def noul_temp(cfg):
    t = cfg.get("temperature", 1.0)
    if isinstance(t, (list, tuple)): return float(t[NOUL])
    if isinstance(t, dict): return float(t.get("noul", 1.0))
    return float(t)

CFGS = {k: json.load(open(os.path.join(d, "rl_agent_config.json"))) for k, d in DIRS.items()}
TEMPS = {k: noul_temp(c) for k, c in CFGS.items()}
WEIGHTS = {k: load_file(os.path.join(d, "model.safetensors")) for k, d in DIRS.items()}
tok = AutoTokenizer.from_pretrained(os.path.join(DIRS["merged"], "tokenizer"))
cfg = CFGS["merged"]
model = build_model(cfg, encoder_dir=os.path.join(DIRS["merged"], "encoder")).to("cuda").eval()
def use(name): model.load_state_dict(WEIGHTS[name], strict=True)
print("temperatures:", {k: round(v, 3) for k, v in TEMPS.items()})

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

temperatures: {'merged': 1.103, 'fine_tuned': 1.241, 'zero_shot': 1.983}


In [5]:
def build_all(texts, labels=LABELS):
    seqs = []
    for i, text in enumerate(texts):
        state = {'comment': str(text)[:MAX_CHARS]}
        for l in labels:
            ids, mk = build_sequence(tok, state, {"t": "noul", "ins": QUESTIONS[l]["instructions"], "crit": {}},
                                     cfg["max_len"], cfg["head_max_len"])
            seqs.append((i, LABELS.index(l), ids, mk))
    return seqs

def collate(items):
    n, L = len(items), max(len(s) for s, _ in items); kmax = max(len(m) for _, m in items)
    ids = torch.full((n, L), tok.pad_token_id, dtype=torch.long); att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long); mmask = torch.zeros((n, kmax), dtype=torch.bool)
    for i, (s, m) in enumerate(items):
        ids[i, :len(s)] = torch.tensor(s); att[i, :len(s)] = 1
        mpos[i, :len(m)] = torch.tensor(m); mmask[i, :len(m)] = True
    return ids.cuda(), att.cuda(), mpos.cuda(), mmask.cuda(), torch.full((n,), NOUL, dtype=torch.long).cuda()

@torch.no_grad()
def score(seqs, n_rows, sort=True):
    order = sorted(range(len(seqs)), key=lambda k: len(seqs[k][2])) if sort else list(range(len(seqs)))
    margin = np.full((n_rows, len(LABELS)), -1e9); finish = np.zeros(n_rows)
    torch.cuda.synchronize(); t0 = time.time()
    for b in range(0, len(order), BATCH):
        idx = order[b:b + BATCH]
        with torch.autocast("cuda", dtype=torch.float16):
            logits, _ = model(*collate([(seqs[k][2], seqs[k][3]) for k in idx]))
        lg = logits.float()[:, :2].cpu().numpy(); now = time.time() - t0
        for r, k in enumerate(idx):
            i, j = seqs[k][0], seqs[k][1]
            margin[i, j] = lg[r, 1] - lg[r, 0]; finish[i] = max(finish[i], now)
    torch.cuda.synchronize()
    return margin, time.time() - t0, finish

sig = lambda x: 1 / (1 + np.exp(-x))
def aucs(y, m):
    a = {l: roc_auc_score(y[:, j], m[:, j]) for j, l in enumerate(LABELS)}; a["mean"] = float(np.mean(list(a.values()))); return a

## 3. Score all three models on the same 10k comments (also the real run for the stream replay)

In [6]:
sample_seqs = build_all(sample.comment_text.tolist())
M, P, SPEED, FINISH = {}, {}, {}, {}
for name in ["zero_shot", "fine_tuned", "merged"]:
    use(name)
    M[name], secs, FINISH[name] = score(sample_seqs, len(sample))
    P[name] = sig(M[name] / TEMPS[name]); SPEED[name] = len(sample) / secs
    print(f"{name}: mean AUC {aucs(ys, M[name])['mean']:.4f} | {SPEED[name]:.1f} comments/sec")

W0926 09:25:24.101000 24 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


zero_shot: mean AUC 0.9718 | 23.1 comments/sec
fine_tuned: mean AUC 0.9748 | 23.5 comments/sec
merged: mean AUC 0.9838 | 23.6 comments/sec


## 4. Calibration: does '80% sure' mean right 80% of the time?

In [7]:
def calib(p, y, bins=10):
    p, y = p.ravel(), y.ravel(); edges = np.linspace(0, 1, bins + 1); rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p >= lo) & (p < hi) if hi < 1 else (p >= lo)
        if m.sum(): rows.append({"bin": f"{lo:.1f}-{hi:.1f}", "mean_pred": float(p[m].mean()), "observed": float(y[m].mean()), "n": int(m.sum())})
    return rows
cal = {}
plt.figure(figsize=(5.5, 5.5)); plt.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
for name in P:
    p, y = P[name].ravel(), ys.ravel()
    ece = ece_score(np.maximum(p, 1 - p), ((p >= 0.5) == (y == 1)).astype(float))
    rows = calib(P[name], ys); cal[name] = {"ece": float(ece), "bins": rows}
    plt.plot([r["mean_pred"] for r in rows], [r["observed"] for r in rows], "o-", label=f"{name} (ECE {ece:.3f})")
    print(f"{name}: ECE {ece:.4f}")
plt.xlabel("Predicted probability"); plt.ylabel("Actual fraction positive"); plt.title("Calibration (10k test comments, 6 labels)")
plt.legend(); plt.tight_layout(); plt.savefig(f"{PLOTS}/calibration.png", dpi=150); plt.show()
json.dump(cal, open(f"{OUT}/calibration.json", "w"), indent=1)
print(pd.DataFrame(cal["merged"]["bins"]).round(3).to_string(index=False))

zero_shot: ECE 0.0635
fine_tuned: ECE 0.0134
merged: ECE 0.0236
    bin  mean_pred  observed     n
0.0-0.1      0.009     0.001 53358
0.1-0.2      0.144     0.068   782
0.2-0.3      0.248     0.083   520
0.3-0.4      0.346     0.084   297
0.4-0.5      0.449     0.092   250
0.5-0.6      0.549     0.135   266
0.6-0.7      0.647     0.150   301
0.7-0.8      0.754     0.213   381
0.8-0.9      0.853     0.220   763
0.9-1.0      0.967     0.593  3082


## 5. Routing: auto-remove / auto-approve / send to human (merged model, 'toxic' question)

In [8]:
full_path = glob.glob('/kaggle/input/**/merged_test_margins.csv', recursive=True)
if full_path:
    fm = pd.read_csv(full_path[0]).values; rset, ry = fm, test[LABELS].values; rtexts = test.comment_text.values
    print("Using full test set margins from Phase 2c:", len(fm))
else:
    rset, ry, rtexts = M["merged"], ys, sample.comment_text.values
    print("Phase 2c output not attached - using the 10k sample")
rp = sig(rset / TEMPS["merged"]); pt, yt = rp[:, 0], ry[:, 0]

rows = []
for lo in [0.02, 0.05, 0.1, 0.2]:
    for hi in [0.8, 0.9, 0.95, 0.98]:
        rem, app = pt >= hi, pt <= lo; hum = ~(rem | app)
        rows.append({"approve_below": lo, "remove_above": hi,
                     "auto_removed_%": 100 * rem.mean(), "auto_approved_%": 100 * app.mean(), "to_human_%": 100 * hum.mean(),
                     "removal_precision_%": 100 * yt[rem].mean() if rem.any() else np.nan,
                     "toxic_missed_in_approved_%": 100 * yt[app].mean() if app.any() else np.nan})
route = pd.DataFrame(rows)
print(route.round(2).to_string(index=False))
json.dump({"p": np.round(pt, 4).tolist(), "y": yt.astype(int).tolist()}, open(f"{OUT}/routing_points.json", "w"))
route.to_json(f"{OUT}/routing_table.json", orient="records", indent=1)

plt.figure(figsize=(6, 4))
for lo in [0.02, 0.05, 0.1]:
    s = route[route.approve_below == lo]; plt.plot(s["to_human_%"], s["removal_precision_%"], "o-", label=f"approve below {lo}")
plt.xlabel("% sent to human review"); plt.ylabel("Precision of auto-removals (%)"); plt.title("Routing trade-off")
plt.legend(); plt.tight_layout(); plt.savefig(f"{PLOTS}/routing.png", dpi=150); plt.show()

Using full test set margins from Phase 2c: 63978
 approve_below  remove_above  auto_removed_%  auto_approved_%  to_human_%  removal_precision_%  toxic_missed_in_approved_%
          0.02          0.80           18.45            66.67       14.88                48.78                        0.05
          0.02          0.90           15.39            66.67       17.94                55.65                        0.05
          0.02          0.95           12.12            66.67       21.21                64.11                        0.05
          0.02          0.98            7.13            66.67       26.20                79.59                        0.05
          0.05          0.80           18.45            71.18       10.37                48.78                        0.11
          0.05          0.90           15.39            71.18       13.43                55.65                        0.11
          0.05          0.95           12.12            71.18       16.70                6

## 6. Error review (what it gets wrong)

In [9]:
yany = ry.max(1)
fp = np.where(yany == 0)[0][np.argsort(-pt[yany == 0])][:15]
fn = np.where(yt == 1)[0][np.argsort(pt[yt == 1])][:15]
clip = lambda t: str(t)[:160].replace("\n", " ")
errs = {"false_positives": [{"p_toxic": round(float(pt[i]), 3), "text": clip(rtexts[i])} for i in fp],
        "false_negatives": [{"p_toxic": round(float(pt[i]), 3), "text": clip(rtexts[i])} for i in fn]}
print("FALSE POSITIVES (flagged, but labeled clean):"); print(pd.DataFrame(errs["false_positives"]).to_string(index=False))
print(); print("FALSE NEGATIVES (labeled toxic, model says clean):"); print(pd.DataFrame(errs["false_negatives"]).to_string(index=False))
json.dump(errs, open(f"{OUT}/errors.json", "w"), indent=1)

FALSE POSITIVES (flagged, but labeled clean):
 p_toxic                                                                                                                                                             text
   0.993                                                      haiasikag pooop onnn brandon mcdoggle berry aisjksrglskedjsgnejhgj fart bag banna saj usejhsedhus gay suase
   0.992                                              @ABOVE YOU SHUT THE F*** UP AND LEAVE MY TALK PAGE ALONE OR ELSE I WILL SEND YOU TO NORTH KOREA!!!!!!!!!!!!!!!!!!!!
   0.992 == Hi ==    F!!!Y!!!S!!!S!!!O!!!D   U!!!O!!!K!!!U!!!U!!!I   C!!!U!!!I!!!C!!!R!!!C   K!!!!!!!T!!!K!!!!!!!K          !T!          !L!          !E!          !S!   
   0.992                                                                           xgay gay gay gay gya gya gya gya gfyagatyyzsgdfsgfuxzkmgtiodfg'rfhg r hbfdj        GAY
   0.992                                                               GET A LIFE KISS A GIRL LICK A CHE

## 7. Bias / robustness: HateCheck (functional tests for hate speech)

In [10]:
hc_ok = True
try:
    from datasets import load_dataset
    ds = load_dataset("Paul/hatecheck"); hc = ds[list(ds.keys())[0]].to_pandas()
    print(len(hc), "cases | columns:", list(hc.columns))
except Exception as e:
    hc_ok = False; print("HateCheck could not be loaded - skipping. Error:", e)

if hc_ok:
    hy = (hc["label_gold"] == "hateful").astype(int).values
    hseqs = build_all(hc["test_case"].tolist(), labels=["identity_hate"])
    res, flags = {}, {}
    for name in ["zero_shot", "merged"]:
        use(name); m, _, _ = score(hseqs, len(hc)); s = m[:, LABELS.index("identity_hate")]
        flags[name] = sig(s / TEMPS[name]) >= 0.5; res[name] = s
    from detoxify import Detoxify
    dtx = Detoxify("original", device="cuda"); texts = hc["test_case"].tolist(); da = []
    for i in range(0, len(texts), 64): da += dtx.predict(texts[i:i+64])["identity_attack"]
    res["detoxify"] = np.array(da); flags["detoxify"] = res["detoxify"] >= 0.5; del dtx

    summ = pd.DataFrame({n: {"AUC (threshold-free)": roc_auc_score(hy, res[n]),
                             "accuracy @0.5": (flags[n] == hy).mean(),
                             "hateful caught (recall)": flags[n][hy == 1].mean(),
                             "non-hateful wrongly flagged": flags[n][hy == 0].mean()} for n in res}).T
    print(summ.round(3).to_string())
    per = pd.DataFrame({n: pd.Series(flags[n] == hy).groupby(hc["functionality"].values).mean() for n in res})
    per["gold"] = hc.groupby("functionality")["label_gold"].first()
    print(); print("Accuracy per test type (worst first, merged):"); print(per.sort_values("merged").round(2).to_string())
    json.dump({"summary": summ.round(4).to_dict(), "per_functionality": per.round(4).reset_index().to_dict("records")},
              open(f"{OUT}/hatecheck.json", "w"), indent=1)

README.md: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/3728 [00:00<?, ? examples/s]

3728 cases | columns: ['functionality', 'case_id', 'test_case', 'label_gold', 'target_ident', 'direction', 'focus_words', 'focus_lemma', 'ref_case_id', 'ref_templ_id', 'templ_id', 'case_templ']
Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to /root/.cache/torch/hub/checkpoints/toxic_original-c1212f89.ckpt


100%|██████████| 418M/418M [00:00<00:00, 501MB/s]


...

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

           AUC (threshold-free)  accuracy @0.5  hateful caught (recall)  non-hateful wrongly flagged
zero_shot                 0.847          0.675                    0.590                        0.138
merged                    0.805          0.677                    0.623                        0.206
detoxify                  0.694          0.533                    0.398                        0.170

Accuracy per test type (worst first, merged):
                    zero_shot  merged  detoxify         gold
spell_space_add_h        0.21    0.34      0.22      hateful
spell_leet_h             0.39    0.39      0.17      hateful
slur_h                   0.70    0.45      0.10      hateful
counter_quote_nh         0.54    0.46      0.68  non-hateful
spell_char_del_h         0.49    0.49      0.27      hateful
spell_space_del_h        0.59    0.55      0.40      hateful
negate_pos_h             0.33    0.56      0.29      hateful
spell_char_swap_h        0.43    0.57      0.28      hateful


## 8. Speed: one-step vs two-step check, GPU vs CPU

In [11]:
use("merged")
# Two-step accuracy, simulated on the routing set: detail questions only if P(toxic) >= tau
flag = rp[:, 0] >= CASCADE_TAU
casc = rset.copy(); casc[~flag, 1:] = -1e9
casc_auc = aucs(ry, casc); full_auc = aucs(ry, rset)
avg_q = 1 + 5 * flag.mean()
print(f"Two-step: {100*flag.mean():.1f}% flagged -> {avg_q:.2f} questions/comment (vs 6)")
print(f"Mean AUC one-step {full_auc['mean']:.4f} | two-step {casc_auc['mean']:.4f}")

# Real two-step timing on 5k comments
sub = sample.iloc[:5000]; torch.cuda.synchronize(); t0 = time.time()
m1, _, _ = score(build_all(sub.comment_text.tolist(), labels=["toxic"]), len(sub))
hit = np.where(sig(m1[:, 0] / TEMPS["merged"]) >= CASCADE_TAU)[0]
seqs2 = build_all(sub.comment_text.iloc[hit].tolist(), labels=LABELS[1:])
if seqs2: score(seqs2, len(hit))
two_step_speed = len(sub) / (time.time() - t0)
print(f"GPU one-step: {SPEED['merged']:.1f} comments/sec | GPU two-step (incl. tokenizing): {two_step_speed:.1f} comments/sec")

# CPU (2 threads, like a free Hugging Face Space), official Agent
torch.set_num_threads(2)
agent_cpu = laya.Agent(DIRS["merged"], device="cpu")
cpu_states = [{'comment': t[:MAX_CHARS]} for t in sample.comment_text.iloc[:20]]
agent_cpu.predict_batch(cpu_states[:2], QUESTIONS, batch_size=8)
t0 = time.time(); agent_cpu.predict_batch(cpu_states, QUESTIONS, batch_size=8); cpu_s = (time.time() - t0) / len(cpu_states)
t0 = time.time(); agent_cpu.predict_batch(cpu_states, {"toxic": QUESTIONS["toxic"]}, batch_size=8); cpu_1q = (time.time() - t0) / len(cpu_states)
print(f"CPU (2 threads): {cpu_s:.2f} s/comment with 6 questions | {cpu_1q:.2f} s/comment toxic-only")
del agent_cpu; torch.set_num_threads(os.cpu_count())
speed = {"gpu_one_step_cps": SPEED["merged"], "gpu_two_step_cps": two_step_speed, "two_step_avg_questions": avg_q,
         "two_step_mean_auc": casc_auc["mean"], "one_step_mean_auc": full_auc["mean"],
         "cpu_sec_per_comment_6q": cpu_s, "cpu_sec_per_comment_1q": cpu_1q}
json.dump(speed, open(f"{OUT}/speed.json", "w"), indent=1)

Two-step: 26.9% flagged -> 2.35 questions/comment (vs 6)
Mean AUC one-step 0.9853 | two-step 0.9841
GPU one-step: 23.6 comments/sec | GPU two-step (incl. tokenizing): 56.6 comments/sec
CPU (2 threads): 7.27 s/comment with 6 questions | 1.01 s/comment toxic-only


## 9. Race vs Gemini (optional)

In [12]:
import requests
race = None
try:
    GEMINI_KEY = secrets.get_secret("GEMINI_API_KEY") if RUN_RACE else None
except Exception:
    GEMINI_KEY = None
if not GEMINI_KEY:
    print("No GEMINI_API_KEY secret (or RUN_RACE=False) - skipping race.")
else:
    rs = sample.sample(RACE_N, random_state=5); use("merged")
    PROMPT = ("Classify this online comment. Reply with JSON only: keys toxic, severe_toxic, obscene, threat, insult, "
              "identity_hate, each a probability between 0 and 1.\n\nComment: ")
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent?key={GEMINI_KEY}"
    race = []
    for text, labs in zip(rs.comment_text, rs[LABELS].values):
        t = str(text)[:MAX_CHARS]
        m, lsec, _ = score(build_all([t]), 1)                      # Laya: one comment, 6 questions, GPU
        lp = sig(m[0] / TEMPS["merged"])
        g = {"ok": False}
        for attempt in range(3):
            t0 = time.time()
            r = requests.post(url, json={"contents": [{"parts": [{"text": PROMPT + t}]}],
                                         "generationConfig": {"temperature": 0, "responseMimeType": "application/json"}}, timeout=120)
            gsec = time.time() - t0
            if r.status_code == 429: time.sleep(30); continue
            try:
                out = json.loads(r.json()["candidates"][0]["content"]["parts"][0]["text"])
                g = {"ok": True, "ms": round(gsec * 1000), "p": [float(out.get(l, 0)) for l in LABELS]}
            except Exception as e:
                g = {"ok": False, "ms": round(gsec * 1000), "error": str(r.status_code)}
            break
        race.append({"text": t[:280], "y": labs.astype(int).tolist(),
                     "laya": {"ms": round(lsec * 1000, 1), "p": np.round(lp, 3).tolist()}, "gemini": g})
        time.sleep(max(0, 60 / GEMINI_RPM - (g.get("ms", 0) / 1000)))   # rate-limit pause, NOT counted in timings
    ok = [x for x in race if x["gemini"]["ok"]]
    print(f"Gemini answered {len(ok)}/{len(race)}")
    if ok:
        print(f"Median latency  Laya {np.median([x['laya']['ms'] for x in race]):.0f} ms | Gemini {np.median([x['gemini']['ms'] for x in ok]):.0f} ms")
        yy = np.array([x["y"][0] for x in ok])
        acc = lambda k: np.mean([(x[k]["p"][0] >= 0.5) == (x["y"][0] == 1) for x in ok])
        print(f"Toxic accuracy on these {len(ok)}: Laya {acc('laya'):.2%} | Gemini {acc('gemini'):.2%} ({yy.sum()} toxic in set)")
    json.dump(race, open(f"{OUT}/race_log.json", "w"))

Gemini answered 22/100
Median latency  Laya 80 ms | Gemini 1840 ms
Toxic accuracy on these 22: Laya 86.36% | Gemini 81.82% (2 toxic in set)


## 10. Stream replay + final scoreboard files

In [13]:
stream = [{"text": str(t)[:280].replace("\n", " "), "y": y.astype(int).tolist(),
           "p": np.round(p, 3).tolist(), "t": round(float(f), 3)}
          for t, y, p, f in zip(sample.comment_text, ys, P["merged"], FINISH["merged"])]
stream.sort(key=lambda x: x["t"])
json.dump({"device": "Tesla T4 (Kaggle, free)", "questions_per_comment": 6, "comments": stream},
          open(f"{OUT}/stream_log.json", "w"))

scoreboard = {"full_test": {
    "TF-IDF + LR": {"toxic": .9643, "severe_toxic": .9831, "obscene": .9770, "threat": .9920, "insult": .9703, "identity_hate": .9850, "mean": .9786},
    "Detoxify original": {"toxic": .9739, "severe_toxic": .9915, "obscene": .9824, "threat": .9965, "insult": .9805, "identity_hate": .9928, "mean": .9863},
    "Laya merged (ours)": {"toxic": .9726, "severe_toxic": .9887, "obscene": .9828, "threat": .9962, "insult": .9813, "identity_hate": .9902, "mean": .9853}},
  "sample_10k": {n: aucs(ys, M[n]) for n in M},
  "ece_sample_10k": {n: cal[n]["ece"] for n in cal}}
json.dump(scoreboard, open(f"{OUT}/scoreboard.json", "w"), indent=1)
for f in sorted(os.listdir(OUT)): print(f, f"{os.path.getsize(os.path.join(OUT, f))/1e6:.2f} MB")

calibration.json 0.00 MB
errors.json 0.00 MB
hatecheck.json 0.00 MB
race_log.json 0.04 MB
routing_points.json 0.70 MB
routing_table.json 0.00 MB
scoreboard.json 0.00 MB
speed.json 0.00 MB
stream_log.json 2.88 MB
